In [663]:
import os
import random
import pandas as pd
import numpy as np
import pyemu

In [664]:
# set up path
target_dir = r"C:\Python\Personal\proj6\codes\sandbox"
rel_t_d = os.path.relpath(target_dir, os.getcwd()) 
os.chdir(rel_t_d)

In [665]:
##########################
### pit-centric domain ###
##########################

# number of wells
n_wells = 2

# number of individuals in the population in the decision variable space
num_reals = 150

# bounds of coordinate system
x_min, x_max = -50, 50
y_min, y_max = -50, 50

# wells cannot be within this many units of the center
no_go_radius = 10.0

# range of pumping rates [m3/day]
q_min, q_max = 6.5, 65

######################
### parameter list ###
######################

# build parameter data frame
par_names = []
for i in range(1, n_wells + 1):
    par_names.extend([f"well_{i}_x", f"well_{i}_y", f"well_{i}_q"])

if "Xc" not in par_names:
    par_names.append("Xc")

#######################
### initialize pest ###
#######################

# initialize an empty PEST control file (v2)
pst = pyemu.pst_utils.generic_pst(par_names=par_names)
pdf = pst.parameter_data

###########################
### populate pest files ###
###########################

# randomize compliance point y-position
Yc = np.random.choice([y_min, y_max])
print(f"QC: Compliance point Y-coordinate set to {Yc}")

# configure well position metadata
well_pars = [p for p in par_names if "well" in p]
pdf.loc[well_pars, ["partrans", "parchglim", "pargp"]] = ["none", "factor", "well_pos"]
pdf.loc[well_pars, ["parlbnd", "parubnd"]] = [x_min, x_max]

# configure pumping rate metadata
well_q_pars = [p for p in par_names if "_q" in p]
pdf.loc[well_q_pars, ["partrans", "parchglim", "pargp"]] = ["none", "factor", "pumping_rate"]
pdf.loc[well_q_pars, "parlbnd"] = q_min
pdf.loc[well_q_pars, "parubnd"] = q_max

# configure compliance point metadata
pdf.loc["Xc", ["partrans", "parchglim", "pargp"]] = ["none", "factor", "compliance"]
pdf.loc["Xc", ["parlbnd", "parubnd"]] = [x_min, x_max]

###################
### random draw ###
###################

# initialize within constraints
valid_data = []
while len(valid_data) < num_reals:
    # generate random values for all parameters at once
    row = np.random.uniform(0, 1, len(par_names)) # start with 0-1 scale
    
    # scale each parameter to its specific bounds
    scaled_row = []
    for p_name, val in zip(par_names, row):
        lb = pdf.loc[p_name, "parlbnd"]
        ub = pdf.loc[p_name, "parubnd"]
        scaled_row.append(lb + val * (ub - lb))
    
    scaled_row = np.array(scaled_row)
    
    # no-go zone check
    # the first 2*n_wells are X and Y
    # skip Q for the distance check
    well_coords = []
    for i in range(1, n_wells + 1):
        well_coords.append([scaled_row[par_names.index(f"well_{i}_x")], 
                            scaled_row[par_names.index(f"well_{i}_y")]])
    
    distances = np.linalg.norm(well_coords, axis=1)
    if np.all(distances > no_go_radius):
        valid_data.append(scaled_row)

pop = pyemu.ParameterEnsemble(pst=pst, df=pd.DataFrame(valid_data, columns=par_names))


# generate random ensemble for PESTPP-MOU
np.random.seed(np.random.randint(1, 100000))
pop = pyemu.ParameterEnsemble(pst=pst, df=pd.DataFrame(valid_data, columns=par_names))

# select a random realization from the initial population by picking one index name at random from the ensemble
Yc = np.random.choice([y_min, y_max])
selected_real = np.random.choice(pop.index)
pst.parameter_data.loc[par_names, "parval1"] = pop.loc[selected_real, par_names].values

###########################
### objective functions ###
###########################

# min Xc
new_obs = {"obsnme": "obj_min_xc", "obsval": 0.0, "weight": 1.0, "obgnme": "min_xc_gp"}
for col, val in new_obs.items():
    pst.observation_data.loc["obj_min_xc", col] = val

# min total Q
new_obs_q = {"obsnme": "obj_min_q", "obsval": 0.0, "weight": 0.1, "obgnme": "min_q_gp"}
for col, val in new_obs_q.items():
    pst.observation_data.loc["obj_min_q", col] = val

########################
### write and record ###
########################

# set control data for PEST control file (v2)
pst.control_data.noptmax = 0

# write PEST control file (v2)
pst_filename = "fwd_model.pst"
pst.write(pst_filename, version = 2)

# record to external file in the current directory
pop.to_csv("initial_pop.csv")

# Create an empty list to store well data
well_summary = []

for i in range(1, n_wells + 1):
    well_summary.append({
        "Well Name": f"Well {i}",
        "X Coordinate": pst.parameter_data.loc[f"well_{i}_x", "parval1"],
        "Y Coordinate": pst.parameter_data.loc[f"well_{i}_y", "parval1"],
        "Pumping Rate (Q)": pst.parameter_data.loc[f"well_{i}_q", "parval1"]
    })

#####################
### summary table ###
#####################

well_summary = []

for i in range(1, n_wells + 1):
    well_summary.append({
        "Well Name": f"Well {i}",
        "X Coordinate": pst.parameter_data.loc[f"well_{i}_x", "parval1"],
        "Y Coordinate": pst.parameter_data.loc[f"well_{i}_y", "parval1"],
        "Pumping Rate (m3/day)": pst.parameter_data.loc[f"well_{i}_q", "parval1"]
    })

df_summary = pd.DataFrame(well_summary)

# Display the table
print(f"Selected Realization: {selected_real}")
display(df_summary.style.format({
    "X Coordinate": "{:.2f}",
    "Y Coordinate": "{:.2f}",
    "Pumping Rate (m3/day)": "{:.2f}"
}))


QC: Compliance point Y-coordinate set to -50
noptmax:0, npar_adj:7, nnz_obs:3
Selected Realization: 118


,Well Name,X Coordinate,Y Coordinate,Pumping Rate (m3/day)
0,Well 1,-15.20,-15.70,47.25
1,Well 2,-32.45,31.66,30.19


In [666]:
import plotly.graph_objects as go
fig = go.Figure()
# access the underlying dataframe using the internal _df attribute
df = pop._df

######################################
### two well example visualization ###
######################################

# add boundary box
fig.add_shape(type="rect", x0=x_min, y0=y_min, x1=x_max, y1=y_max, 
              line=dict(color="black"),
              opacity=0.2
)

# open pit
fig.add_trace(go.Scatter(
    x=[0], y=[0], mode='markers+text', text=["Open Pit"],
    marker=dict(color='black', size=15, symbol='hexagon'),
    name="Open Pit (Datum)",
    textposition="top center"
))

# no-go zone
fig.add_shape(type="circle",
    xref="x", yref="y",
    x0=-no_go_radius, y0=-no_go_radius, x1=no_go_radius, y1=no_go_radius,
    line=dict(color="black", dash="dash")
)

# well 1 ensemble
fig.add_trace(go.Scatter(
    x=df["well_1_x"], y=df["well_1_y"],
    mode='markers', 
    marker=dict(color='lightskyblue', opacity=0.3, symbol='circle'),
    name="Well 1 Ensemble"
))

# well 1 selected
fig.add_trace(go.Scatter(
    x=[pst.parameter_data.loc["well_1_x", "parval1"]], 
    y=[pst.parameter_data.loc["well_1_y", "parval1"]],
    mode='markers+text', text=["Well 1"], textposition="top center",
    marker=dict(color='dodgerblue', size=10, symbol='circle', 
                line=dict(width=1, color='dodgerblue')),
    name="Well 1 Selected (parval1)"
))

# well 2 ensemble
fig.add_trace(go.Scatter(
    x=df["well_2_x"], y=df["well_2_y"],
    mode='markers', 
    marker=dict(color='lightcoral', opacity=0.3, symbol='square'),
    name="Well 2 Ensemble"
))

# well 2 selected
fig.add_trace(go.Scatter(
    x=[pst.parameter_data.loc["well_2_x", "parval1"]], 
    y=[pst.parameter_data.loc["well_2_y", "parval1"]],
    mode='markers+text', text=["Well 2"], textposition="top center",
    marker=dict(color='crimson', size=10, symbol='square', 
                line=dict(width=1, color='crimson')),
    name="Well 2 Selected (parval1)"
))

# compliance point
fig.add_trace(go.Scatter(
    x=[pst.parameter_data.loc["Xc", "parval1"]], 
    y=[Yc], 
    mode='markers+text', 
    text=["Compliance"], 
    # flip text position based on which edge it is on
    textposition="top center" if Yc == y_min else "bottom center",
    marker=dict(
        color='yellow', 
        size=15, 
        symbol='star', 
        line=dict(width=1, color='black')
    ),
    name=f"Compliance (Y={Yc})"
))

fig.update_layout(
    xaxis=dict(range=[x_min-5, x_max+5], title="X Coordinate"),
    yaxis=dict(range=[y_min-5, y_max+5], title="Y Coordinate"),
    title=f"Well Locations (Realization: {selected_real} of {num_reals})",
    template="plotly_white",
    width=700, height=700
)

fig.show()
